In [ ]:
from crewai import LLM, Agent, Task, Crew
from dotenv import load_dotenv
load_dotenv()

# Agents

In [ ]:
import os

In [ ]:

llm = LLM(
    model="openai/gpt-5-nano",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0.2,
)

In [ ]:
agent = Agent(
    role = "You are an English to Bengali translator",
    goal = "Translate user's question into Bengali",
    backstory="You have been trained in multiple languages and can fluently converse in multiple languages",
    llm = llm,
    verbose = True,
    max_iter = 5,
    allow_delegation = True,
    cache=True,
    # system_template=,
    # prompt_template=,
    max_retry_limit=5,
)

In [ ]:
response = await agent.kickoff("What is the favourite fish among Bengali people?")
response

# Tasks

In [ ]:
task = Task(
    # name = "Translator",
    description= "Translate user question",
    expected_output="Translated text",
    agent=agent,
    # output_json=,
    # context=,
    # tools=,
    # callback=,
    # guardrails=,
    # guardrail_max_retries=3,
    # config=
)

In [ ]:
await task.aexecute_sync()

# Crews

In [ ]:

crew = Crew(
    tasks=[task],
    agents=[agent],
    # process = "sequential"
    # manager_agent,
    # config=,
    # memory=,
    # cache=,
    # task_callback=,
    # stream=,
    # after_kickoff_callbacks=,
    # checkpoint=
    # tracing=
)

# Flows

In [ ]:
from crewai.flow.flow import Flow, start, listen
from pydantic import BaseModel

In [ ]:
class wf_state(BaseModel):

    thing1: int
    var2: str


class Workflow(Flow[wf_state]):

    @start
    def entry_node(self):
        pass

    @listen()
    def node2(self):
        self.state.thing1 = 2


# Memory

In [ ]:
# Agent, crew or flow level

# MCP

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters

class MarketAgent:
    def __init__(self):
        self._adapter = MCPServerAdapter(
            StdioServerParameters(
                command="uvx",
                args=["nsekit-mcp"],
            )
        )

        self._tools = self._adapter.__enter__()

        self.agent = Agent(
            role="Indian Equity Researcher",
            goal="Analyze NSE stocks",
            backstory="Expert in Indian equities.",
            tools=self._tools,
            verbose=True,
            llm=llm
        )

    def kickoff(self, prompt: str):
        task = Task(
            description=prompt,
            expected_output="A detailed answer.",
            agent=self.agent,
        )

        crew = Crew(
            agents=[self.agent],
            tasks=[task],
        )

        return crew.kickoff()

    async def kickoff_async(self, prompt: str):
        task = Task(
            description=prompt,
            expected_output="A detailed answer.",
            agent=self.agent,
        )

        crew = Crew(
            agents=[self.agent],
            tasks=[task],
            memory=True
        )

        return await crew.kickoff_async()

    def close(self):
        self._adapter.__exit__(None, None, None)

    def __del__(self):
        self.close()

In [ ]:
market_agent = MarketAgent()

result = await market_agent.kickoff_async(
    "What is the best performing stock today?"
)

print(result)

market_agent.close()

# Threadsafe